# 003 · Args and Kwargs

> **Syntax › Functions** · Python 3.14

`*args` collects any number of extra positional arguments, and `**kwargs` collects any number of extra keyword arguments. Together they let a function accept "anything", which is the backbone of wrappers, decorators and flexible APIs.

**In this notebook you will learn:**

- `*args` and how it compares to C# `params T[]`
- `**kwargs`, which has no direct C# equivalent
- how to combine them with regular, default and keyword-only parameters
- unpacking sequences and dicts at the call site with `*` and `**`
- forwarding arguments through wrapper functions
- how to type-hint `*args` and `**kwargs`, including `Unpack[TypedDict]`

## C# mental model

| C# | Python |
|---|---|
| `params int[] numbers` | `*numbers` (arrives as a `tuple`) |
| (none) arbitrary named arguments | `**options` (arrives as a `dict`) |
| `Sum(1, 2, 3)` or `Sum(array)` | `total(1, 2, 3)` or `total(*lst)` |
| options object / `Dictionary<string, object>` parameter | `**kwargs` |
| `params int[]` with typed elements | `*args: int` (each element is an `int`) |

The names `args` and `kwargs` are only conventions; the `*` and `**` are what matter.

## `*args`: variable positional arguments

```csharp
// C#
static int Sum(params int[] numbers) => numbers.Sum();

Sum();          // 0
Sum(1, 2, 3);   // 6
```

In Python, a parameter prefixed with `*` gathers all remaining positional arguments into a **tuple**:

In [ ]:
def total(*numbers):
    print(f"numbers = {numbers!r} ({type(numbers).__name__})")
    return sum(numbers)


print(total())
print(total(1, 2, 3))

Regular parameters can come before `*args`. Anything **after** `*args` becomes keyword-only, which is the same rule as the bare `*` from *Functions › 002 · Parameters and Arguments*:

In [ ]:
def log(level, *messages, sep=" | "):
    print(f"[{level}] " + sep.join(messages))


log("INFO", "started", "port 8080")
log("WARN", "disk", "85%", sep=" - ")      # sep can only be passed by name

## `**kwargs`: variable keyword arguments

A parameter prefixed with `**` gathers all **unmatched keyword arguments** into a `dict`. C# has no syntax for this; the closest equivalents are passing a `Dictionary<string, object>` or an options object.

```csharp
// C#: the nearest equivalents
static string BuildTag(string name, Dictionary<string, string> attributes) { ... }
BuildTag("a", new() { ["href"] = "/home", ["title"] = "Home" });

// or an options record
record TagOptions(string? Href = null, string? Title = null);
```

In [ ]:
def build_tag(name, **attributes):
    print(f"attributes = {attributes!r}")
    attrs = "".join(f' {key}="{value}"' for key, value in attributes.items())
    return f"<{name}{attrs}>"


print(build_tag("a", href="/home", title="Home"))
print(build_tag("br"))

Keyword names must be valid identifiers, so `class="x"` (a reserved word) or `data-id="1"` cannot be written directly. A common workaround is a trailing underscore that you strip, or unpacking a dict (shown below).

## Combining everything

The complete parameter order is: positional-only, `/`, normal, `*args`, keyword-only, `**kwargs`.

In [ ]:
def everything(a, b=2, *args, flag=False, **kwargs):
    print(f"a={a} b={b} args={args} flag={flag} kwargs={kwargs}")


everything(1)
everything(1, 20, 30, 40)
everything(1, 20, 30, flag=True, color="red", size=3)

⚠️ **Gotcha:** `**kwargs` only receives names that do **not** match a declared parameter. A misspelled keyword goes silently into `kwargs` instead of raising an error, so validate kwargs if typos matter.

In [ ]:
def configure(*, verbose=False, **extra):
    print(f"verbose={verbose} extra={extra}")


configure(verbose=True)
configure(verbos=True)     # typo! no error, just lands in extra

## Unpacking at the call site

The same `*` and `**` symbols do the **opposite** job when you *call* a function: they spread a sequence into positional arguments and a dict into keyword arguments *(see Collections › 006 · Unpacking)*.

C# can only do this for `params` arrays (`Sum(array)`), and never for named arguments.

In [ ]:
def describe(name, qty, price):
    return f"{qty} x {name} @ {price}"


values = ["pen", 3, 1.5]
options = {"name": "cup", "qty": 2, "price": 4.0}

print(describe(*values))         # describe("pen", 3, 1.5)
print(describe(**options))       # describe(name="cup", qty=2, price=4.0)
print(describe("mug", *[1, 9.5]))

You can mix several unpackings with literal arguments. This is how you pass keyword names that are not valid identifiers:

In [ ]:
defaults = {"href": "#", "title": "link"}
overrides = {"title": "Docs"}

print(build_tag("a", **defaults, **{"data-id": "7", "class": "nav"}))
print(build_tag("a", **(defaults | overrides)))    # merge first, later keys win

try:
    build_tag("a", **defaults, **overrides)        # same key twice is an error
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Forwarding arguments (wrappers)

The most important use of `*args, **kwargs` is a function that wraps another and passes **all** arguments through unchanged, whatever its signature. This is the foundation of decorators *(see Functions › 006 · Decorators)*.

In C# you would need a separate overload or generic for every signature; in Python one wrapper fits all.

In [ ]:
def call_logged(func, *args, **kwargs):
    print(f"calling {func.__name__} with args={args} kwargs={kwargs}")
    result = func(*args, **kwargs)     # forward everything
    print(f"  -> {result!r}")
    return result


call_logged(describe, "pen", 3, price=1.5)
call_logged(max, 4, 9, 2)
ordered = call_logged(sorted, ["b", "a", "c"], reverse=True)

Forwarding is also handy for "extend the base call" helpers. Here we inject a default without hiding any other option:

In [ ]:
def sorted_desc(iterable, **kwargs):
    kwargs.setdefault("reverse", True)
    return sorted(iterable, **kwargs)


words = ["kiwi", "fig", "banana"]
print(sorted_desc(words))
print(sorted_desc(words, key=len))
print(sorted_desc(words, key=len, reverse=False))

## Type hints for `*args` and `**kwargs`

Annotate the type of **each element**, not the container. `*args: int` means `args` is a `tuple[int, ...]`; `**kwargs: str` means `kwargs` is a `dict[str, str]`. Hints are not enforced at runtime.

In [ ]:
def average(*values: float) -> float:
    return sum(values) / len(values)


def headers(**fields: str) -> dict[str, str]:
    return {key.replace("_", "-").title(): value for key, value in fields.items()}


print(average(1, 2, 3, 4))
print(headers(content_type="text/plain", accept="*/*"))

When each keyword has its **own** type, describe them with a `TypedDict` and annotate `**kwargs: Unpack[...]` (`typing.Unpack`, Python 3.12+). Type checkers then validate names and types precisely, much like a C# options record. At runtime it is still just a dict.

In [ ]:
from typing import NotRequired, TypedDict, Unpack


class RequestOptions(TypedDict):
    timeout: float
    retries: NotRequired[int]


def fetch(url: str, **options: Unpack[RequestOptions]) -> str:
    retries = options.get("retries", 0)
    return f"GET {url} timeout={options['timeout']} retries={retries}"


print(fetch("/api/orders", timeout=2.5))
print(fetch("/api/orders", timeout=2.5, retries=3))

⚠️ **Gotcha:** `*args, **kwargs` hides the real signature from your IDE and from readers. Prefer explicit parameters in normal application code, and reach for `*args`/`**kwargs` when you genuinely need variadic input or transparent forwarding.

## Summary

| Task | C# | Python |
|---|---|---|
| Variable positional args | `params int[] xs` | `*xs` (a `tuple`) |
| Variable named args | options object / dictionary | `**kw` (a `dict`) |
| Spread a collection into args | `Sum(array)` (only for `params`) | `f(*seq)` |
| Spread a dictionary into named args | not possible | `f(**mapping)` |
| Forward all arguments | one overload per signature | `def w(*args, **kwargs): return f(*args, **kwargs)` |
| Typed variadics | `params int[]` | `*args: int`, `**kwargs: str` |
| Precisely typed options | options record | `**kwargs: Unpack[SomeTypedDict]` |

## Exercises

1. Write `join_all(*parts, sep=", ")` that joins any number of values (converted with `str`) using `sep`.
2. Write `make_record(**fields)` that returns the fields sorted by key as a list of `(key, value)` tuples, then call it by unpacking an existing dict.
3. Write `retry(func, *args, attempts=3, **kwargs)` that calls `func(*args, **kwargs)` up to `attempts` times, returning the first successful result and re-raising the last exception otherwise.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def join_all(*parts, sep=", "):
    return sep.join(str(p) for p in parts)

print(join_all(1, "two", 3.0, sep=" | "))

# 2
def make_record(**fields):
    return sorted(fields.items())

data = {"name": "alice", "age": 30}
print(make_record(**data))

# 3 — keyword-only "attempts" cannot collide with func's positional args
def retry(func, *args, attempts=3, **kwargs):
    last_error = None
    for _ in range(attempts):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_error = e
    raise last_error
```

</details>